In [ ]:
import collections
import collections.abc

# แพตช์ (Patch) เพื่อให้โค้ดเก่า (PrivMRF) ทำงานได้บน Python 3.10+
collections.Iterable = collections.abc.Iterable
collections.Mapping = collections.abc.Mapping
collections.MutableSet = collections.abc.MutableSet
collections.MutableMapping = collections.abc.MutableMapping

import matplotlib
matplotlib.use('Agg') 

import warnings
warnings.filterwarnings('ignore')

import os
import csv
import json
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from sklearn.isotonic import IsotonicRegression # 🌟 เพิ่มตัวคุมกราฟสมูท

try:
    from cuml.svm import SVC
    from cuml.preprocessing import StandardScaler as cuStandardScaler
    from cuml.pipeline import make_pipeline as cu_make_pipeline
    import cupy as cp
    print("🚀 [System] สำเร็จ! โหลด RAPIDS cuML เรียบร้อย (SVM จะรันบน GPU)")
except ImportError:
    from sklearn import svm
    from sklearn.svm import SVC
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline
    print("⚠️ [System] ไม่พบ RAPIDS cuML จะสลับไปใช้ scikit-learn (SVM จะรันบน CPU)")

from exp.evaluate import run_syn 
from PrivMRF.domain import Domain
import PrivMRF.attribute_hierarchy as ah

# ==========================================
# 1. 🎯 ฝั่งเข้ารหัส: DP Noisy CDF Adaptive Binning (Isotonic Enhanced)
# ==========================================
def dp_microbin_cdf_merging(raw_col_data, epsilon, num_micro_bins=2000, target_bins=10):
    domain_min = np.min(raw_col_data)
    domain_max = np.max(raw_col_data)
    
    # 1. ซอยข้อมูลเป็น Micro-bins จิ๋วๆ
    raw_counts, micro_edges = np.histogram(
        raw_col_data, bins=num_micro_bins, range=(domain_min, domain_max)
    )
    
    # 2. ฉีด Laplace Noise (ปล่อยให้ติดลบได้)
    noise_scale = 1.0 / epsilon if epsilon > 0 else 0
    noisy_counts = raw_counts + np.random.laplace(loc=0.0, scale=noise_scale, size=num_micro_bins)
    
    # 3. เกลี่ย Noise (Smoothing) ด้วย Convolution
    kernel_size = 5
    kernel = np.ones(kernel_size) / kernel_size
    smoothed_counts = np.convolve(noisy_counts, kernel, mode='same')
    
    # 🌟 4. [อัปเกรด] บังคับให้ CDF พุ่งขึ้นเสมอด้วย Isotonic Regression 
    # (ช่วยแก้ปัญหาตะกร้ากระโดดไปมาที่ Epsilon = 0.1, 0.2 ได้เด็ดขาด)
    raw_cdf = np.cumsum(smoothed_counts)
    iso = IsotonicRegression(increasing=True, out_of_bounds='clip')
    cdf = iso.fit_transform(np.arange(num_micro_bins), raw_cdf)
    
    total_noisy_pop = cdf[-1]
    if total_noisy_pop <= 0:
        total_noisy_pop = 1.0 

    # 5. หาจุดตัดตะกร้า (Bin Edges)
    target_capacity = total_noisy_pop / target_bins
    adaptive_edges_raw = [domain_min]
    
    current_target = target_capacity
    for i in range(num_micro_bins):
        if cdf[i] >= current_target:
            adaptive_edges_raw.append(micro_edges[i+1])
            current_target += target_capacity
            
    if adaptive_edges_raw[-1] < domain_max:
        adaptive_edges_raw.append(domain_max)
        
    adaptive_edges_raw = np.sort(np.unique(adaptive_edges_raw))
    
    # 6. สร้าง Output Format และคำนวณ Stats 
    adaptive_edges = []
    for i in range(len(adaptive_edges_raw) - 1):
        adaptive_edges.append((adaptive_edges_raw[i], adaptive_edges_raw[i+1]))
        
    bin_stats = {}
    for i in range(len(adaptive_edges)):
        start_val, end_val = adaptive_edges[i]
        in_bin = raw_col_data[(raw_col_data >= start_val) & (raw_col_data <= end_val)]
        
        if len(in_bin) > 0:
            bin_stats[i] = {'mean': np.mean(in_bin), 'std': np.std(in_bin) + 1e-5}
        else:
            bin_stats[i] = {'mean': (start_val + end_val)/2.0, 'std': (end_val - start_val)/4.0}

    return adaptive_edges, len(adaptive_edges), bin_stats

# ==========================================
# 2. 🔄 ฝั่งถอดรหัส: Inverse Transform (Mean Reconstruction)
# ==========================================
def inverse_transform_adaptive(discrete_data, bin_info, num_continuous_cols):
    continuous_data = np.zeros(discrete_data.shape, dtype=float)
    
    for col in range(num_continuous_cols):
        stats = bin_info[col]['stats']
        edges = bin_info[col]['edges']
        col_indices = discrete_data[:, col].astype(int)
        
        safe_indices = np.clip(col_indices, 0, len(edges) - 1)
        
        for row_idx, bin_idx in enumerate(safe_indices):
            continuous_data[row_idx, col] = stats[bin_idx]['mean']
                
    continuous_data[:, -1] = discrete_data[:, -1]
    return continuous_data

# ==========================================
# 3. ไปป์ไลน์หลัก: Ultimate Adaptive CDF Binning 
# ==========================================
def run_adaptive_cdf_pipeline():
    DATA_NAME = 'avila_combined_df'
    CONTINUOUS_COLS = list(range(10))
    LABEL_COL = [-1]
    
    # TARGET_BINS_LIST = [2,3,4,5,6,7,8,9,10]
    TARGET_BINS_LIST = [7,8,9,10]


    # 🌟 กำหนด Epsilon รวมตามมาตรฐาน (เพื่อเทียบ Standard Binning)
    TOTAL_EPS_LIST = [0.1, 0.2, 0.4, 0.8, 1.6, 3.2]
    
    TRAIN_SAMPLE_LIMIT = 10000 
    RAW_FILE_PATH = f'./data/{DATA_NAME}.csv'

    print("🚀 เริ่มต้นระบบ: Ultimate DP Adaptive CDF Binning (Isotonic + Budget Split)")
    print("-" * 85)

    if not os.path.exists(RAW_FILE_PATH):
        print(f"❌ ไม่พบไฟล์ {RAW_FILE_PATH}")
        return
        
    with open(RAW_FILE_PATH, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        headings = next(reader)
        while len(headings) == 0:
            headings = next(reader)
        raw_data = [row for row in reader if len(row) > 0]

    all_headings = [headings[i] for i in CONTINUOUS_COLS] + [headings[LABEL_COL[0]]]

    cont_data = [[float(row[i]) for i in CONTINUOUS_COLS] for row in raw_data] 
    raw_continuous_np = np.array(cont_data)
    raw_label_np = np.array([[row[LABEL_COL[0]]] for row in raw_data]) 
    
    le_master = LabelEncoder()
    encoded_label_master = le_master.fit_transform(raw_label_np.ravel()).reshape(-1, 1)
    raw_scaled = np.concatenate([raw_continuous_np, encoded_label_master], axis=1)

    # --- THE ABSOLUTE TRTR BASELINE ---
    print(f"\n🔥 กำลังประเมิน The Absolute TRTR Baseline (Continuous Space)...")
    np.random.seed(42)
    shuffled_idx_real = np.random.permutation(len(raw_scaled))
    shuffled_raw_scaled = raw_scaled[shuffled_idx_real]
    
    fold_size_real = len(shuffled_raw_scaled) // 5
    trtr_fold_acc, trtr_fold_f1 = [], []
    TARGET_LABEL_INDEX = -1

    for j in range(5):
        start_r, end_r = j * fold_size_real, (j + 1) * fold_size_real
        if j == 4: end_r = len(shuffled_raw_scaled)
        
        X_real_test = shuffled_raw_scaled[start_r:end_r]
        X_real_train_full = np.concatenate([shuffled_raw_scaled[:start_r], shuffled_raw_scaled[end_r:]], axis=0)
        
        if len(X_real_train_full) > TRAIN_SAMPLE_LIMIT:
            np.random.seed(42 + j)
            sample_idx = np.random.choice(len(X_real_train_full), TRAIN_SAMPLE_LIMIT, replace=False)
            X_real_train = X_real_train_full[sample_idx]
        else:
            X_real_train = X_real_train_full

        y_train_trtr = X_real_train[:, TARGET_LABEL_INDEX].astype(int)
        X_train_trtr = np.delete(X_real_train, TARGET_LABEL_INDEX, axis=1)
        
        y_test_trtr = X_real_test[:, TARGET_LABEL_INDEX].astype(int)
        X_test_trtr = np.delete(X_real_test, TARGET_LABEL_INDEX, axis=1)

        clf_trtr = make_pipeline(StandardScaler(), SVC(gamma='auto'))
        clf_trtr.fit(X_train_trtr, y_train_trtr)
        y_pred_trtr = clf_trtr.predict(X_test_trtr)
        
        if hasattr(y_pred_trtr, 'get'): y_pred_trtr = y_pred_trtr.get()
        elif hasattr(y_pred_trtr, 'to_numpy'): y_pred_trtr = y_pred_trtr.to_numpy()
        
        trtr_fold_acc.append(accuracy_score(y_test_trtr, y_pred_trtr))
        trtr_fold_f1.append(f1_score(y_test_trtr, y_pred_trtr, average='macro'))

    absolute_trtr_acc = np.mean(trtr_fold_acc)
    absolute_trtr_f1 = np.mean(trtr_fold_f1)
    print(f"✅ Absolute TRTR ยืนพื้นสำเร็จ! | Acc: {absolute_trtr_acc:.4f} | F1: {absolute_trtr_f1:.4f}")
    print("-" * 85)

    if not os.path.exists('./preprocess'): os.mkdir('./preprocess')
    if not os.path.exists('./out'): os.mkdir('./out')
    if not os.path.exists('./result'): os.mkdir('./result')

    summary_results = []

    # --- BEGIN ADAPTIVE BINNING LOOP ---
    for TARGET_BINS in TARGET_BINS_LIST:
        print(f"\n" + "🔥"*40)
        print(f"🎯 [ทดสอบ Adaptive CDF (เป้าหมายตะกร้า) = {TARGET_BINS}]")
        print("🔥"*40)

        for TOTAL_EPS in TOTAL_EPS_LIST:
            
            # 🌟 แบ่งงบอัตโนมัติ: Binning 20% | PrivMRF 80%
            EPS_BINNING = TOTAL_EPS * 0.5
            EPS_MRF = TOTAL_EPS * 0.5
            
            print(f"\n📦 [Total Epsilon = {TOTAL_EPS:.2f}] กำลังทำงาน...")
            print(f"   -> 💰 จัดสรรให้ PrivMRF (70%): {EPS_MRF:.3f}")
            print(f"   -> 💰 จัดสรรให้ Binning (30%): {EPS_BINNING:.3f}")

            json_domain = {}
            processed_data = np.zeros((len(cont_data), len(all_headings)), dtype=int)
            bin_info = {}
            total_bins_generated = 0

            # 1. รัน Adaptive CDF Binning ทีละคอลัมน์
            for col in range(len(CONTINUOUS_COLS)):
                col_values = raw_continuous_np[:, col]
                
                edges, num_bins_created, stats = dp_microbin_cdf_merging(
                    col_values, 
                    epsilon=EPS_BINNING, 
                    num_micro_bins=2000, # เพิ่มความละเอียด
                    target_bins=TARGET_BINS
                )
                
                total_bins_generated += num_bins_created
                bin_info[col] = {'edges': edges, 'num_bins': num_bins_created, 'stats': stats}
                json_domain[col] = {'domain': num_bins_created}
                
                edge_starts = [e[0] for e in edges]
                b_idx = np.searchsorted(edge_starts, col_values, side='right') - 1
                b_idx = np.clip(b_idx, 0, num_bins_created - 1)
                
                processed_data[:, col] = b_idx

            processed_data[:, -1] = encoded_label_master.ravel()
            json_domain[len(CONTINUOUS_COLS)] = {'domain': len(le_master.classes_)}
            
            avg_bins = total_bins_generated / len(CONTINUOUS_COLS)

            with open(f'./preprocess/{DATA_NAME}.csv', 'w', newline='', encoding='utf-8') as f:
                csv.writer(f).writerow(all_headings)
                csv.writer(f).writerows(processed_data)
            with open(f'./preprocess/{DATA_NAME}.json', 'w', encoding='utf-8') as f:
                json.dump(json_domain, f)

            domain_obj = Domain(json_domain, list(range(len(all_headings))))
            dummy_hierarchy = ah.get_one_level_hierarchy(domain_obj)
            ah.write_hierarchy(dummy_hierarchy, f'./preprocess/{DATA_NAME}_hierarchy.json')

            exp_name = f"AdaptiveCDF_{TARGET_BINS}_Eps_{TOTAL_EPS:.3f}"
            
            # 2. สร้างข้อมูลสังเคราะห์ (PrivMRF) โดยใชังบ 80%
            syn_discrete = run_syn(DATA_NAME, exp_name, epsilon=EPS_MRF, task='TVD')
            syn_discrete = np.array(syn_discrete, dtype=int)
            
            num_cont_cols = len(CONTINUOUS_COLS)
            
            # ถอดรหัส (Inverse Transform) 
            syn_continuous = inverse_transform_adaptive(syn_discrete, bin_info, num_cont_cols)

            # 3. ประเมิน TRUE TSTR
            np.random.seed(42)
            
            shuffled_idx_real = np.random.permutation(len(raw_scaled))
            shuffled_real_raw = raw_scaled[shuffled_idx_real] 
            
            shuffled_idx_syn = np.random.permutation(len(syn_continuous))
            shuffled_syn_continuous = syn_continuous[shuffled_idx_syn]
            
            fold_size_real = len(shuffled_real_raw) // 5
            fold_size_syn = len(shuffled_syn_continuous) // 5
            
            tstr_fold_acc, tstr_fold_f1 = [], []
            
            for j in range(5): 
                start_r, end_r = j * fold_size_real, (j + 1) * fold_size_real
                if j == 4: end_r = len(shuffled_real_raw)
                X_real_test_fold = shuffled_real_raw[start_r:end_r]
                
                start_s, end_s = j * fold_size_syn, (j + 1) * fold_size_syn
                X_syn_train_fold_full = np.concatenate([shuffled_syn_continuous[:start_s], shuffled_syn_continuous[end_s:]], axis=0)
                
                if len(X_syn_train_fold_full) > TRAIN_SAMPLE_LIMIT:
                    np.random.seed(42 + j)
                    sample_indices = np.random.choice(len(X_syn_train_fold_full), TRAIN_SAMPLE_LIMIT, replace=False)
                    X_syn_train_fold = X_syn_train_fold_full[sample_indices]
                else:
                    X_syn_train_fold = X_syn_train_fold_full

                y_train = X_syn_train_fold[:, TARGET_LABEL_INDEX].astype(int)
                X_train = np.delete(X_syn_train_fold, TARGET_LABEL_INDEX, axis=1)
                
                y_test = X_real_test_fold[:, TARGET_LABEL_INDEX].astype(int)
                X_test = np.delete(X_real_test_fold, TARGET_LABEL_INDEX, axis=1)
                
                if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
                    continue
                    
                clf_tstr = make_pipeline(StandardScaler(), SVC(gamma='auto'))
                clf_tstr.fit(X_train, y_train)
                y_pred = clf_tstr.predict(X_test)
                
                if hasattr(y_pred, 'get'): y_pred = y_pred.get()
                elif hasattr(y_pred, 'to_numpy'): y_pred = y_pred.to_numpy()
                
                tstr_fold_acc.append(accuracy_score(y_test, y_pred))
                tstr_fold_f1.append(f1_score(y_test, y_pred, average='macro'))

            tstr_acc_final = np.mean(tstr_fold_acc) if tstr_fold_acc else 0.0
            tstr_f1_final = np.mean(tstr_fold_f1) if tstr_fold_f1 else 0.0

            print(f"   -> [Target Bins={TARGET_BINS:<2} | Eps={TOTAL_EPS:.2f}] TRUE TSTR Acc: {tstr_acc_final:.4f} | F1: {tstr_f1_final:.4f} | Avg Bins = {avg_bins:.1f}")

            summary_results.append({
                'Target Bins': TARGET_BINS,
                'Total Epsilon': TOTAL_EPS,
                'Epsilon MRF': EPS_MRF,
                'Epsilon Binning': EPS_BINNING,
                'Avg Bins Found': avg_bins,
                'Downstream Acc (TSTR)': tstr_acc_final,
                'Downstream Macro F1 (TSTR)': tstr_f1_final,
                'Absolute TRTR Acc': absolute_trtr_acc,
                'Absolute TRTR F1': absolute_trtr_f1,
                'Utility Score (Acc)': (tstr_acc_final / absolute_trtr_acc) if absolute_trtr_acc > 0 else 0.0,
                'Utility Score (F1)': (tstr_f1_final / absolute_trtr_f1) if absolute_trtr_f1 > 0 else 0.0
            })

    print("\n" + "🔥"*45)
    print("🎯 สรุปผล: Ultimate Adaptive CDF Binning (Isotonic + Budget Split)")
    print("🔥"*45)
    
    summary_df = pd.DataFrame(summary_results)
    summary_df.sort_values(by=['Target Bins', 'Total Epsilon'], inplace=True)
    
    for col in ['Avg Bins Found', 'Downstream Acc (TSTR)', 'Downstream Macro F1 (TSTR)', 'Absolute TRTR Acc', 'Absolute TRTR F1', 'Utility Score (Acc)', 'Utility Score (F1)']:
        summary_df[col] = summary_df[col].apply(lambda x: f"{x:.5f}" if isinstance(x, float) else x)
    
    print(summary_df.to_string(index=False))
    print("-" * 85)
    
    summary_df.to_csv('./result/ultimate_adaptive_cdf_results.csv', index=False)
    print("✅ รายงานผลถูกเซฟลง ./result/ultimate_adaptive_cdf_results.csv เรียบร้อยแล้วครับ!")

if __name__ == '__main__':
    run_adaptive_cdf_pipeline()